# habituation-vad: re-lit training features for gate G2 (Kaggle)

The legitimate change of phase B becomes a **controlled illumination change**: the same
training videos re-rendered as `dim` (dusk), `night`, `warm` (tungsten light) and `bright`
(sunlight), see `src/preprocess/appearance.py`. Only the **training** split and **2×2** regions,
rgb + motion: 8 new feature folders. The original features are not recomputed.

**Before running** (same as the first extraction notebook)
1. Settings → Accelerator → **GPU T4**. Settings → **Internet on**.
2. Add Input → your private dataset with the **updated** `habituation-vad` repository (zip the folder again: it must contain `src/preprocess/appearance.py`).
3. Add Input → ShanghaiTech Campus, **both parts** (`nikanvasei/shanghaitech-campus-dataset` and `nikanvasei/shanghaitech-campus-dataset-test`).

Then *Save Version → Save & Run All*. Expected time: roughly half to two-thirds of the first extraction.
Download `features_shanghaitech_relit.zip` and `appearance_preview.jpg` from the Output panel.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install open_clip_torch==2.24.0
import torch; print("torch", torch.__version__, "cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), (
    "NO GPU. Right panel -> Session options -> Accelerator -> GPU T4 x2, then Run All again. "
    "(On CPU each video takes minutes instead of seconds.)")

In [ ]:
# copy the repository from /kaggle/input to a writable place
import glob, os, shutil, sys
hits = glob.glob('/kaggle/input/**/src/preprocess/extract.py', recursive=True)
assert hits, 'repository not found: add the habituation-vad dataset as an input'
src_repo = hits[0].rsplit('/src/preprocess/', 1)[0]
REPO = '/kaggle/working/habituation-vad'
if not os.path.exists(REPO):
    shutil.copytree(src_repo, REPO)
os.chdir(REPO); sys.path.insert(0, REPO)
print('repo:', REPO)
assert os.path.exists(f'{REPO}/src/preprocess/appearance.py'), (
    'old repository zip: re-zip the folder so it contains src/preprocess/appearance.py')


In [ ]:
# Collect ShanghaiTech Campus from ALL inputs and sort videos by their OFFICIAL names:
#   official training videos are named 01_001 (3 digits)  -> normal only, go into memory
#   official test videos are named     01_0014 (4 digits) -> evaluated with frame labels
# This matters because some Kaggle copies use a re-split that puts official test
# videos (with anomalies) into a "train" folder. Each video is linked into
#   /kaggle/working/sht/training/..., testing/..., testing/test_frame_mask/
import re, shutil, numpy as np
TRAIN_RE = re.compile(r'^\d{2}_\d{3}$')
TEST_RE = re.compile(r'^\d{2}_\d{4}$')
IMG, VID = ('.jpg', '.jpeg', '.png', '.bmp'), ('.avi', '.mp4')
EXPECTED = {'train': 330, 'test': 107}          # official ShanghaiTech Campus totals

def is_frame_dir(p):
    for e in os.scandir(p):
        return e.name.lower().endswith(IMG)
    return False

def better(new, old):      # prefer a path that says "frames" (not optical flow etc.)
    return 'frame' in new.lower() and 'frame' not in old.lower()

src = {'train': {}, 'test': {}}
labels = {}
for d, dirnames, files in os.walk('/kaggle/input', followlinks=True):
    keep = []
    for n in dirnames:
        split = 'train' if TRAIN_RE.match(n) else 'test' if TEST_RE.match(n) else None
        if split is None:
            keep.append(n); continue
        p = os.path.join(d, n)
        if is_frame_dir(p) and (n not in src[split] or better(p, src[split][n][1])):
            src[split][n] = ('frames', p)
    dirnames[:] = keep                          # never walk inside a video's frames
    for f in files:
        stem, ext = os.path.splitext(f)
        split = 'train' if TRAIN_RE.match(stem) else 'test' if TEST_RE.match(stem) else None
        if split and ext.lower() in VID:
            src[split].setdefault(stem, ('videos', os.path.join(d, f)))
        elif split == 'test' and ext == '.npy' and stem not in labels:
            if np.load(os.path.join(d, f), mmap_mode='r').ndim == 1:   # frame labels, not pixel masks
                labels[stem] = os.path.join(d, f)

for split in ('train', 'test'):
    kinds = {k for k, _ in src[split].values()}
    print(f'{split}: {len(src[split])} videos found (official: {EXPECTED[split]}), stored as {kinds or "-"}')
print(f'frame labels found for {sum(v in labels for v in src["test"])} / {len(src["test"])} test videos')

missing_lab = sorted(v for v in src['test'] if v not in labels)
assert len(src['train']) == EXPECTED['train'] and len(src['test']) == EXPECTED['test'], (
    'Incomplete dataset. Add BOTH Kaggle parts as inputs: '
    'nikanvasei/shanghaitech-campus-dataset AND nikanvasei/shanghaitech-campus-dataset-test')
assert not missing_lab, f'test videos without frame labels: {missing_lab[:10]}'
for split in ('train', 'test'):
    assert len({k for k, _ in src[split].values()}) == 1, f'{split}: mix of frame folders and video files'

DATA = '/kaggle/working/sht'
shutil.rmtree(DATA, ignore_errors=True)
for split, top in (('train', 'training'), ('test', 'testing')):
    for name, (kind, p) in src[split].items():
        link = f'{DATA}/{top}/{kind}/{os.path.basename(p)}'
        os.makedirs(os.path.dirname(link), exist_ok=True)
        os.symlink(p, link)
os.makedirs(f'{DATA}/testing/test_frame_mask')
for name in src['test']:
    os.symlink(labels[name], f'{DATA}/testing/test_frame_mask/{name}.npy')
# frame file naming: text sort and numeric sort must agree (the code sorts numerically)
kind, p = next(iter(src['test'].values()))
if kind == 'frames':
    names = [e.name for e in os.scandir(p)]
    num = sorted(names, key=lambda n: [int(t) if t.isdigit() else t for t in re.split(r'(\d+)', n)])
    print('frame names look like:', num[:3], '...', num[-1],
          '| zero-padded' if sorted(names) == num else '| NOT zero-padded (handled by numeric sort)')
print('data_root:', DATA)

In [ ]:
# write a Kaggle config from the shared one: same settings, Kaggle paths
import yaml
cfg = yaml.safe_load(open('configs/shanghaitech.yaml'))
cfg['data_root'] = DATA
cfg['features_root'] = '/kaggle/working/features'
SCENES = None                      # None = all 13 scenes; or e.g. ['04', '05']
cfg['scenes'] = SCENES
yaml.safe_dump(cfg, open('configs/kaggle.yaml', 'w'), sort_keys=False)
print(open('configs/kaggle.yaml').read())
print('outputs per video:', cfg['extract'])

In [ ]:
# what the four lighting conditions look like (one training frame)
import cv2, numpy as np
from src.preprocess.appearance import PRESETS, apply_appearance, resolve_spec
from pathlib import Path
from src.preprocess.datasets import iter_frames, list_videos
item = list_videos(cfg['dataset'], Path(DATA), 'train', ['01'])[0]
frame = next(iter(iter_frames(item)))
tiles = [frame] + [apply_appearance(frame, resolve_spec(n), item.video_id, 0) for n in PRESETS]
for t, name in zip(tiles, ['original'] + list(PRESETS)):
    cv2.putText(t, name, (12, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 0), 3)
small = [cv2.resize(t, (428, 240)) for t in tiles] + [np.zeros((240, 428, 3), np.uint8)]
grid = np.vstack([np.hstack(small[:3]), np.hstack(small[3:])])
cv2.imwrite('/kaggle/working/appearance_preview.jpg', cv2.cvtColor(grid, cv2.COLOR_RGB2BGR))
from IPython.display import Image; Image('/kaggle/working/appearance_preview.jpg')


In [ ]:
# 1-minute smoke test: 2 training videos, all four lighting conditions
!python -m src.preprocess.extract --config configs/kaggle.yaml --split train --grids 2 --appearances dim night warm bright --limit 2


In [ ]:
# full pass: training split, 2x2 regions, rgb + motion, 4 lighting conditions (8 folders).
# Restartable: outputs already written for a video are skipped.
!python -m src.preprocess.extract --config configs/kaggle.yaml --split train --grids 2 --appearances dim night warm bright


In [ ]:
# check: every folder has all 330 training videos
import pandas as pd
root = f"{cfg['features_root']}/{cfg['dataset']}"
for d in sorted(os.listdir(root)):
    n_npy = len(glob.glob(f'{root}/{d}/train/*.npy'))
    n_csv = pd.read_csv(f'{root}/{d}/train_clips.csv').video_id.nunique()
    print(f'{d:32s} {n_npy} npy, {n_csv} videos in index', '' if n_npy == n_csv == 330 else '<-- INCOMPLETE')


In [ ]:
# package for download (unzip into capstone-pw27-pac04/features/ next to the existing folders)
import shutil
shutil.make_archive('/kaggle/working/features_shanghaitech_relit', 'zip', '/kaggle/working/features')
print(os.listdir('/kaggle/working'))
